## Part 3: Feature Engineering

The goal of this notebook is to generate features for use in training predictive models using variables contained in the dataset. 

This will involve: 
- Regularizing features so that they are all roughly the same magnitude.
- Calculating the percent change of any features that have a strong trend over time. For these types of features, the magnitude of the change from month to month can have very different importance depending on its current value. For example, QQQ increasing from $1 to $2 is much more interesting than it increasing from $100 to $101.
- Defining the target variables for various predictors I want to train. 
    - In the case of defining market health, I will consider two target variables:
        - the GDP next quarter
        - the GDP in a year
    - I will also train a model to predict the value of QQQ in a month's time and see if I can use that to create an QQQ buy strategy that outcompetes a simple buy-and-hold strategy. That work will be done in the 4th notebook.

--------

Notes to self:

Okay, so I think I have a decent start here, but I have more work to do with the actual feature engineering.

Basically, I jumped passed that and made some models that seem to work surprisingly better than I would have guessed.
But now I should take a more serious look at the actual features and correct them as needed.
- Those that consistently increase (cpi, for eg) should most likely be converted to pct_change since it is often their rate of change that is more important anyways (quickly increasing cpi means high inflation)
- Those that hover around a mean can likely just be z-scored and they will be functional then.
- Those that have min value 0 but go up every now and then can prob be scaled so that their max is 1.
- 

After that, I can move all of the actual model building stuff to the next workbook. And do all of the model evaluation things.

---

In [47]:
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from macro_etf.common import PROCESSED_DATA_DIR, MODELLING_TARGETS

data_fn = "processed_market_data.csv"


### 3.1: Load data

In [48]:
df = pd.read_csv(PROCESSED_DATA_DIR / data_fn, index_col=0)
df.index = pd.to_datetime(df.index)
df.head()

,yield_spread,credit_spread,financial_stress,initial_claims,cpi,unemployment,fed_funds_rate,industrial_production,retail_sales,consumer_sentiment,personal_savings_rate,housing_starts,building_permits,m2,job_openings,credit_card_delinquency,close_spy,close_qqq,close_vix,close_oil,volume_spy,volume_qqq,volume_oil,real_gdp
2001-02-28,0.51,2.88,0.437625,371250.0,175.6,4.2,5.49,91.9020,247339.0,94.7,4.5,1600.0,1699.0,4978.3,5088.0,4.81,78.485298,39.983036,28.350000,27.420000,14825800.0,102187900.0,82364.0,14229.765
2001-03-31,0.75,3.04,0.675540,387200.0,176.0,4.2,5.31,91.3034,247289.0,90.6,4.6,1625.0,1656.0,5017.0,5234.0,4.81,74.087158,32.989159,28.639999,26.400000,9183600.0,63661300.0,63947.0,14183.120
2001-04-30,1.05,2.73,0.800700,396750.0,176.1,4.3,4.80,91.1162,244514.0,91.5,4.9,1590.0,1659.0,5074.8,5097.0,4.81,80.417152,38.887619,25.480000,28.459999,10766900.0,73859100.0,40223.0,14183.120
2001-05-31,1.21,2.65,0.387725,394500.0,176.4,4.4,4.21,90.7891,249113.0,88.4,4.8,1649.0,1666.0,5139.1,4762.0,4.94,79.966370,37.691078,22.639999,28.370001,9874200.0,68018300.0,101694.0,14183.120
2001-06-30,1.17,2.65,0.420820,397200.0,177.3,4.3,3.97,90.3555,250250.0,92.0,4.3,1605.0,1665.0,5137.2,4615.0,4.94,78.060844,38.508430,19.059999,26.250000,9824200.0,59719600.0,86303.0,14271.694


### 3.2: Define a feature matrix and prediction targets

In [49]:
# Let's start by defining the feature matrix as the entire unaltered feature set
X = df.copy()

for target, target_param in MODELLING_TARGETS.items():
    X[f'target_{target}'] = (
        X[target_param['variable']]
            .pct_change(periods=target_param['date_gap']) #Take the percent change as that is more informative for variables with a trend.
            .shift(periods=-target_param['date_gap']) #shift the variables so that they correspond to the date gap specified
    )

The features should be normalized / regularized / modified before being passed as training variables to a model. For example, those that trend upwards over time (e.g., cpi, retail sales, etc) can be handled poorly depending on choice of model. Also for these, what is often more important is the percent change of these features over time rather than just the raw number, e.g., an ETF increasing in value from 1 to 2 is a lot more interesting than an increase of 100 to 101. So, I will instead calculate the percent changes of these.

In [50]:
# The number of months between recordings of different features. gdp is quarterly and so the percent change will have to be over 3 months
trending_features_lags = {
    'cpi': 1,
    'retail_sales': 1,
    'm2': 1,
    'close_spy': 1,
    'close_qqq': 1,
    'close_oil': 1,
    'real_gdp': 3
}

for feat, lag in trending_features_lags.items():
    X[f'{feat}_percent_change'] = X[feat].pct_change(periods=lag)
    X = X.drop(feat, axis=1)

In [51]:
# Due to calculating percent changes and shifting, there will now be some nan entries at the end of the dataset. Let's remove them.
X = X.dropna()

In [52]:
# Display the results
display(X.head())
display(X.tail())

,yield_spread,credit_spread,financial_stress,initial_claims,unemployment,fed_funds_rate,industrial_production,consumer_sentiment,personal_savings_rate,housing_starts,building_permits,job_openings,credit_card_delinquency,close_vix,volume_spy,volume_qqq,volume_oil,target_gdp_3mo,target_gdp_12mo,target_qqq_1mo,target_qqq_3mo,cpi_percent_change,retail_sales_percent_change,m2_percent_change,close_spy_percent_change,close_qqq_percent_change,close_oil_percent_change,real_gdp_percent_change
2001-05-31,1.21,2.65,0.387725,394500.0,4.4,4.21,90.7891,88.4,4.8,1649.0,1666.0,4762.0,4.94,22.639999,9874200.0,68018300.0,101694.0,0.006245,0.013373,0.021686,-0.181087,0.001704,0.018809,0.012670,-0.005606,-0.030769,-0.003162,-0.003278
2001-06-30,1.17,2.65,0.420820,397200.0,4.3,3.97,90.3555,92.0,4.3,1605.0,1665.0,4615.0,4.94,19.059999,9824200.0,59719600.0,86303.0,-0.004006,0.013254,-0.086214,-0.365865,0.005102,0.004564,-0.000370,-0.023829,0.021686,-0.074727,0.006245
2001-07-31,1.28,2.78,0.408925,398000.0,4.5,3.77,89.8784,92.6,4.2,1636.0,1626.0,4425.0,4.94,21.620001,11918100.0,55732500.0,52462.0,-0.004006,0.013254,-0.122845,-0.188219,0.002256,-0.005650,0.008351,-0.010196,-0.086214,0.003810,0.006245
2001-08-31,1.21,2.90,0.360120,398000.0,4.6,3.65,89.3028,92.4,5.4,1670.0,1598.0,4361.0,5.00,24.920000,15985400.0,54309000.0,47548.0,-0.004006,0.013254,-0.208845,0.082446,-0.001688,-0.006028,0.005772,-0.059332,-0.122845,0.032258,0.006245
2001-09-30,1.74,3.42,1.194475,435000.0,4.9,3.07,89.2003,91.5,6.5,1567.0,1615.0,4447.0,5.00,31.930000,21687200.0,84810800.0,60080.0,0.002748,0.021465,0.169773,0.342650,0.000000,0.009113,0.006488,-0.081630,-0.208845,-0.138603,-0.004006


,yield_spread,credit_spread,financial_stress,initial_claims,unemployment,fed_funds_rate,industrial_production,consumer_sentiment,personal_savings_rate,housing_starts,building_permits,job_openings,credit_card_delinquency,close_vix,volume_spy,volume_qqq,volume_oil,target_gdp_3mo,target_gdp_12mo,target_qqq_1mo,target_qqq_3mo,cpi_percent_change,retail_sales_percent_change,m2_percent_change,close_spy_percent_change,close_qqq_percent_change,close_oil_percent_change,real_gdp_percent_change
2025-02-28,0.25,1.57,-0.701125,225000.0,4.0,4.33,100.0647,71.7,5.1,1353.0,1463.0,7295.0,3.06,19.629999,88744100.0,47654000.0,250074.0,-0.001625,0.019893,-0.075862,0.023052,0.004273,-0.010564,0.002900,-0.012695,-0.027035,-0.038191,0.004599
2025-03-31,0.34,1.76,-0.453250,223000.0,4.2,4.33,101.0993,64.7,5.2,1491.0,1446.0,7431.0,3.06,22.280001,95328200.0,53000300.0,313087.0,0.009460,0.026847,0.013968,0.177727,0.002251,0.002424,0.003030,-0.055719,-0.075862,0.024656,-0.001625
2025-04-30,0.57,1.96,0.060925,225750.0,4.2,4.33,101.0404,57.0,5.1,1346.0,1492.0,7242.0,3.06,24.700001,93101500.0,46810600.0,419549.0,0.009460,0.026847,0.091783,0.189654,0.000332,0.014245,0.003711,-0.008670,0.013968,-0.185646,-0.001625
2025-05-31,0.52,1.84,-0.543680,231800.0,4.2,4.33,101.1279,52.2,5.5,1400.0,1445.0,6952.0,3.04,18.570000,90601200.0,67662800.0,384927.0,0.009460,0.026847,0.063858,0.100038,0.001617,-0.001836,0.003789,0.062845,0.091783,0.044322,-0.001625
2025-06-30,0.52,1.75,-0.731225,239000.0,4.3,4.33,100.9655,52.2,4.9,1289.0,1416.0,7098.0,3.04,16.730000,92502500.0,45548700.0,193289.0,0.010763,0.017224,0.024237,0.089598,0.000993,-0.012386,0.002668,0.051386,0.063858,0.071064,0.009460


### 3.3 Split the data into training and test sets

In [54]:
# Let's split the data into training and test data
train_prop = 0.7
test_prop = 1.0 - train_prop
n_train = int(train_prop*X.shape[0])
n_test = X.shape[0] - n_train
train_inds = np.arange(0,n_train)
test_inds = np.arange(n_train, X.shape[0])


split_labels = ['train']*n_train + ['test']*n_test
X['split'] = split_labels

### 3.4: Regularize the features

In [ ]:
from sklearn.preprocessing import StandardScaler

# Train a scaler using the training dataset and then apply the scaler to both the training and test datasets
#   We do this split so that information contained in the training set isn't accidentally incorporated into the test set.
#   This can happen when calculating a feature's z-score if all values are used to calculate the mean and std.
#   I also need to be careful to not scale the target variables, as otherwise I will have to de-scale them and that would be annoying

target_cols = [col for col in X.columns if 'target' in col] + ['split']
non_target_cols = list (set(X.columns) - set(target_cols))

X_train = X.query('split == "train"')[non_target_cols]
X_test = X.query('split == "test"')[non_target_cols]
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = pd.DataFrame(
    scaler.transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

X_scaled = pd.concat(
    [
        X_train_scaled, 
        X_test_scaled
    ], 
    axis=0
)
X_scaled = pd.concat(
    [X_scaled, X[target_cols]], axis=1
)

In [56]:
print(target_cols)
print(non_target_cols)

['target_gdp_3mo', 'target_gdp_12mo', 'target_qqq_1mo', 'target_qqq_3mo', 'split']
['close_qqq_percent_change', 'yield_spread', 'personal_savings_rate', 'fed_funds_rate', 'close_vix', 'cpi_percent_change', 'unemployment', 'financial_stress', 'close_oil_percent_change', 'consumer_sentiment', 'initial_claims', 'credit_spread', 'building_permits', 'close_spy_percent_change', 'real_gdp_percent_change', 'job_openings', 'volume_spy', 'housing_starts', 'volume_oil', 'industrial_production', 'volume_qqq', 'm2_percent_change', 'retail_sales_percent_change', 'credit_card_delinquency']


### 3.5: Save training / test datasets + target variables

In [57]:
#I'm going to try to keep things pandasy and output one big dataframe containing the required info.

# save_df_X = pd.concat([
#     X_train_scaled,
#     X_test_scaled
# ])

# save_df_y = pd.concat([
#     pd.DataFrame(y_train).assign(split='train'),
#     pd.DataFrame(y_test).assign(split='test')
# ])

# save_df = pd.concat([
#     save_df_X,
#     save_df_y,
# ], axis=1)

# save_df.to_csv(PROCESSED_DATA_DIR / 'feature_matrix.csv')
X_scaled.to_csv(PROCESSED_DATA_DIR / 'feature_matrix.csv')